# 36 · DenseNet121 congelado: cinco pliegues de desarrollo

**Revisión 1.0. Esta libreta sí ejecuta entrenamiento, no es otra prueba de dos pasos.** Selecciona **GPU** y ejecuta las celdas en orden en una sesión nueva. El único modelo es **DenseNet121**, con su **codificador congelado** y la cabeza de clasificación entrenable, semilla **2026**, en los cinco pliegues ya establecidos de desarrollo.

Cada pliegue comienza desde el checkpoint público original verificado y una cabeza nueva. No se utilizan los pesos de la prueba breve 35. Se aplica la receta registrada: máximo de **60 épocas**, parada temprana con paciencia **8**, micro-lote de **2** y grupo efectivo de **16** rodillas. Los cinco pliegues se ejecutan en secuencia; esta etapa puede durar varias horas según la GPU y la parada temprana. No se entrenan otros modelos, no se descongela el codificador, no se cambian las particiones y **no se abre la prueba reservada**.

La libreta verifica la prueba GPU 35 ya guardada y copia únicamente el paquete aprobado de desarrollo. No se modifica ningún original. La normalización y el procesamiento clínico se limitan a lo ya registrado; no se incorporan variables o imágenes nuevas.

**Cada pliegue completo se guarda y verifica en Drive antes de iniciar el siguiente.** Si se pierde la sesión, vuelve a abrir esta misma revisión y ejecuta todas las celdas desde el inicio, con las mismas carpetas. Se recuperan únicamente los pliegues completos, autenticados y compatibles con el código y los datos exactos. Un pliegue interrumpido se reinicia desde el checkpoint público; no se continúan su optimizador ni sus pesos parciales. No borres las carpetas de resultados.

**No ejecutes dos sesiones o copias de esta libreta simultáneamente sobre las mismas carpetas.** Cada campaña admite un solo proceso escritor. Las publicaciones finales se preparan en archivos temporales únicos y solo reciben su nombre final después de verificar todas sus huellas. Un archivo temporal interrumpido se conserva, pero no se usa como resultado completo.

Necesitarás las mismas dos carpetas de Drive: registros privados y desarrollo (la que contiene `transport.json` y `model_inputs_loading_reference_v1.json`). No selecciones el paquete reservado. Los pesos y las predicciones se guardan únicamente en la carpeta privada de Drive, nunca en GitHub.

Los resultados de esta etapa describen validación interna de desarrollo. No constituyen una evaluación final, una comparación con los demás modelos o una validación clínica.

In [ ]:
from pathlib import Path, PurePosixPath
import json, os, re, shutil, stat, subprocess, sys, tempfile, urllib.request, zipfile

CODE_REVISION = '257b5f7a656c75b5d45f612acb31eb48e911a4ad'
if not re.fullmatch(r'[0-9a-f]{40}', CODE_REVISION) or CODE_REVISION == '0' * 40:
    raise RuntimeError('Esta copia no tiene una revisión publicada. Usa la libreta final entregada.')
if os.environ.get('COLAB_GPU', '0') != '1' or any(os.environ.get(k) for k in ('COLAB_TPU_ADDR', 'TPU_NAME', 'COLAB_TPU_TASK_ID')):
    raise RuntimeError('Selecciona GPU en Entorno de ejecución > Cambiar tipo de entorno de ejecución. No uses CPU ni TPU.')
if not os.environ.get('COLAB_RELEASE_TAG'):
    raise RuntimeError('Esta libreta debe ejecutarse personalmente en Google Colab; no es un ensayo local.')
if 'knee' in sys.modules or any(name.startswith('knee.') for name in sys.modules):
    raise RuntimeError('Reinicia el entorno para cargar la revisión fijada de esta libreta.')
SESSION_ROOT = Path(tempfile.mkdtemp(prefix='knee_densenet36_'))
REPO_DIR = SESSION_ROOT / 'repository'
archive_path = SESSION_ROOT / 'code.zip'
url = 'https://codeload.github.com/AlonsoYess/Knee/zip/' + CODE_REVISION
with urllib.request.urlopen(url, timeout=60) as response, archive_path.open('xb') as target:
    if response.geturl() != url:
        raise RuntimeError('La descarga del código cambió de ubicación.')
    total = 0
    while True:
        chunk = response.read(1024**2)
        if not chunk:
            break
        total += len(chunk)
        if total > 128 * 1024**2:
            raise RuntimeError('El archivo del repositorio supera el límite autorizado.')
        target.write(chunk)
REPO_DIR.mkdir()
with zipfile.ZipFile(archive_path) as archive:
    members = archive.infolist()
    if not members or len(members) > 20000 or sum(i.file_size for i in members) > 512 * 1024**2:
        raise RuntimeError('El contenido del repositorio supera los límites de extracción.')
    seen = set()
    for item in members:
        name = item.filename.rstrip('/')
        parts = name.split('/')
        if not name or '\\' in name or ':' in name or '\0' in name or any(p in ('', '.', '..') for p in parts) or parts[0] != 'Knee-' + CODE_REVISION or name in seen or stat.S_ISLNK(item.external_attr >> 16):
            raise RuntimeError('Se encontró una ruta no permitida en el repositorio.')
        seen.add(name)
        destination = REPO_DIR.joinpath(*parts[1:])
        if item.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        else:
            if len(parts) < 2:
                raise RuntimeError('El archivo del repositorio no tiene una ruta relativa válida.')
            destination.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as source, destination.open('xb') as target:
                shutil.copyfileobj(source, target, length=1024**2)
sys.path.insert(0, str(REPO_DIR / 'src'))
from knee import densenet_fivefold_colab as orchestration

def checked_run(arguments, **kwargs):
    result = subprocess.run(arguments, capture_output=True, text=True, **kwargs)
    if result.returncode:
        detail = re.sub(r'https?://\S+', '<url>', (result.stdout + result.stderr)[-12000:])
        detail = re.sub(r'(?i)(authorization|token|password|secret)\s*[:=]\s*\S+', r'\1=<redacted>', detail)
        print(detail)
        raise RuntimeError('La ejecución se detuvo. Conserva el error y los pliegues completos de Drive; no cambies la receta.')
    return result.stdout

print('Revisión exacta del código preparada. No se abrieron datos de participantes.')


## 1. Preparar el intérprete GPU aislado

Se utiliza el mismo entorno CUDA verificado en la libreta 35. La GPU real y las versiones se comprueban antes de abrir datos. No se modifica el intérprete principal de Colab ni se permite sustituir silenciosamente GPU por CPU.

In [ ]:
print('Preparando el intérprete GPU aislado; aún no se abrieron datos ni se entrenó…')
runtime = json.loads(checked_run([sys.executable, '-B', str(REPO_DIR / 'scripts/train_densenet_fivefold_colab.py'), '--operation', 'setup', '--repository-root', str(REPO_DIR), '--destination', str(SESSION_ROOT / 'gpu_runtime')]))
NEURAL_PYTHON = Path(runtime['python_path'])
RUN_ENV = orchestration.runtime_environment(REPO_DIR)
print('Intérprete y GPU verificados. Todavía no se entrenó ningún pliegue.')


## 2. Verificar el código mediante pruebas sintéticas

En este subproceso se oculta la GPU. Las pruebas emplean datos sintéticos y sustituciones de las operaciones de entrenamiento; no entrenan modelos reales ni abren el paquete de participantes. Cualquier error u omisión detiene la libreta.

In [ ]:
TEST_ENV = dict(RUN_ENV)
TEST_ENV.update(CUDA_VISIBLE_DEVICES='', COLAB_GPU='0')
CPU_TESTS = json.loads(checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/train_densenet_fivefold_colab.py'), '--operation', 'tests', '--repository-root', str(REPO_DIR)], cwd=REPO_DIR, env=TEST_ENV))
print(f"Pruebas sintéticas aprobadas: {CPU_TESTS['tests_run']}; omitidas: {CPU_TESTS['skipped']}.")


## 3. Localizar la prueba 35 y reconstruir solo desarrollo

Indica las dos carpetas habituales. Se autentican los registros completos de la prueba GPU 35 con revisión AMP corregida y se reconstruye únicamente el transporte de desarrollo. Las referencias, partes y huellas se comprueban sin modificar las fuentes de Drive. No es necesario repetir las libretas anteriores.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PERSISTENT_ROOT = orchestration.checked_path(input('Carpeta existente de KneeOAI con los registros privados de la libreta 35: ').strip())
REMOTE_PACKAGE_DIR = orchestration.checked_path(input('Misma carpeta privada de desarrollo, que contiene transport.json: ').strip())
for selected in (PERSISTENT_ROOT, REMOTE_PACKAGE_DIR):
    if not selected.is_dir() or not selected.is_relative_to(Path('/content/drive')):
        raise RuntimeError('Selecciona carpetas existentes dentro de tu Drive montado.')
proof = json.loads(checked_run([str(NEURAL_PYTHON), '-B', str(REPO_DIR / 'scripts/train_densenet_fivefold_colab.py'), '--operation', 'locate', '--repository-root', str(REPO_DIR), '--records-root', str(PERSISTENT_ROOT)], cwd=REPO_DIR, env=RUN_ENV))
SMOKE_DIR = Path(proof['smoke_dir'])
LOCAL_PACKAGE_DIR = SESSION_ROOT / 'development'
REFERENCE_DIR = SESSION_ROOT / 'reference'
REFERENCE_DIR.mkdir(exist_ok=False)
LOCAL_REFERENCE = REFERENCE_DIR / 'model_inputs_loading_reference_v1.json'
transport = orchestration.stage_development(REMOTE_PACKAGE_DIR, LOCAL_PACKAGE_DIR, LOCAL_REFERENCE)
print('Prueba 35 autenticada. Solo desarrollo reconstruido y verificado; originales intactos.')


## 4. Confirmar y entrenar únicamente DenseNet121 congelado

Escribe **`AUTORIZO DENSENET 5 PLIEGUES`** para autorizar una sola ejecución de la receta mostrada en los cinco pliegues de desarrollo. No se solicitará otra confirmación entre pliegues.

Mantén la sesión conectada y permite que el proceso termine. Se mostrará cuándo empieza y termina cada pliegue y una señal periódica de que el subproceso sigue vivo, con tiempo transcurrido real; **esa señal no representa un porcentaje de avance**. No se inventa un progreso de épocas ni una hora de finalización. Los pliegues terminados se copian a Drive y se verifican inmediatamente. Una reconexión no retoma un pliegue a medias.

El codificador permanece congelado. AMP conserva las mismas rodillas si debe reducir su escala, sin descartar ejemplos. Si hay un error de memoria u otro fallo, conserva el mensaje y los resultados ya guardados; no modifiques el lote, las épocas o el modelo por tu cuenta.

In [ ]:
confirmation = input('Escribe AUTORIZO DENSENET 5 PLIEGUES para ejecutar solo la etapa descrita: ').strip()
if confirmation != 'AUTORIZO DENSENET 5 PLIEGUES':
    raise RuntimeError('No se confirmó la etapa. No se inició entrenamiento.')
import queue, threading, time, uuid

def training_run(arguments, *, cwd, env):
    # Stream real stderr; stdout is the structured final record, not a progress estimate.
    process = subprocess.Popen(arguments, cwd=cwd, env=env, text=True, bufsize=1,
        stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    messages = queue.Queue()
    def forward(stream, channel):
        try:
            for line in iter(stream.readline, ''):
                messages.put((channel, line))
        finally:
            stream.close()
            messages.put((channel, None))
    for stream, channel in ((process.stdout, 'stdout'), (process.stderr, 'stderr')):
        threading.Thread(target=forward, args=(stream, channel), daemon=True).start()
    started, last_heartbeat, closed = time.monotonic(), time.monotonic(), set()
    stdout_lines, error_lines = [], []
    try:
        while len(closed) < 2:
            try:
                channel, line = messages.get(timeout=1)
            except queue.Empty:
                channel, line = None, None
            if channel is not None:
                if line is None:
                    closed.add(channel)
                elif channel == 'stdout':
                    stdout_lines.append(line)
                else:
                    safe = re.sub(r'https?://\S+', '<url>', line.rstrip())
                    safe = re.sub(r'(?i)(authorization|token|password|secret)\s*[:=]\s*\S+', r'\1=<redacted>', safe)
                    error_lines.append(safe)
                    if len(error_lines) > 100:
                        error_lines.pop(0)
                    print(safe, flush=True)
            now = time.monotonic()
            if now - last_heartbeat >= 30 and process.poll() is None:
                print(f'Subproceso activo; tiempo transcurrido: {int(now - started)} s. No es un porcentaje de avance.', flush=True)
                last_heartbeat = now
        status = process.wait()
    except BaseException:
        process.terminate()
        process.wait(timeout=30)
        raise
    if status:
        raise RuntimeError('El entrenamiento se detuvo. Conserva el error y los pliegues completos guardados en Drive; no cambies la receta.')
    return ''.join(stdout_lines)

WEIGHTS_ROOT = SESSION_ROOT / 'public_weight_cache'
WEIGHTS_ROOT.mkdir(exist_ok=False)
LOCAL_OUTPUT = SESSION_ROOT / ('fivefold_' + uuid.uuid4().hex)
TEST_RECEIPT = SESSION_ROOT / 'synthetic_test_receipt.json'
RUNTIME_RECEIPT = SESSION_ROOT / 'gpu_setup_receipt.json'
orchestration._save(TEST_RECEIPT, CPU_TESTS)
orchestration._save(RUNTIME_RECEIPT, runtime)
print('Entrenando DenseNet121 con codificador congelado, cinco pliegues de desarrollo y semilla 2026. Se guardará cada pliegue antes de continuar…', flush=True)
training_log = training_run([str(NEURAL_PYTHON), '-u', '-B', str(REPO_DIR / 'scripts/train_densenet_fivefold_colab.py'), '--operation', 'train', '--repository-root', str(REPO_DIR), '--smoke-dir', str(SMOKE_DIR), '--package-dir', str(LOCAL_PACKAGE_DIR), '--reference-path', str(LOCAL_REFERENCE), '--weights-root', str(WEIGHTS_ROOT), '--output-dir', str(LOCAL_OUTPUT), '--records-root', str(PERSISTENT_ROOT), '--code-revision', CODE_REVISION, '--test-receipt', str(TEST_RECEIPT), '--runtime-receipt', str(RUNTIME_RECEIPT), '--approval-statement', orchestration.APPROVAL_STATEMENT], cwd=REPO_DIR, env=RUN_ENV)
FINAL_CHECK = json.loads((LOCAL_OUTPUT / orchestration.SUMMARY_FILENAME).read_text(encoding='utf-8'))
orchestration.validate_result(FINAL_CHECK)
print('Los cinco pliegues terminaron; los resultados privados ya están guardados y verificados en Drive.')


## 5. Mostrar el registro final

La selección interna se resume mediante la media de las métricas de los cinco pliegues; las predicciones agrupadas se describen por separado. Esta etapa no determina aún cuál es el mejor modelo entre todas las alternativas. No se ajustan probabilidades finales, umbrales o modelos con la prueba reservada.

Comparte únicamente el **JSON final**. No compartas imágenes, pesos privados ni identificadores de participantes.

In [ ]:
print(json.dumps(FINAL_CHECK, ensure_ascii=False, indent=2))
print('DenseNet121 congelado: cinco pliegues de desarrollo guardados y verificados. Prueba reservada sin abrir; no se entrenaron otros modelos ni se descongeló el codificador.')
